# Sydney Bank – Customer Churn: Decision Tree & k-Nearest Neighbours
DSB102 Assessment 2 (Classification theme)

This notebook builds the **baseline**, a **Decision Tree** and a **k-Nearest Neighbours (KNN)** model, and evaluates them
the way the business will actually use them: the retention team can only contact the **top 10% most at-risk customers**.

Run top to bottom (`Runtime → Run all`).

## 1. Setup

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, roc_curve, confusion_matrix, ConfusionMatrixDisplay)

RANDOM_STATE = 42          # fixed so every run gives the same numbers as the report
CONTACT_SHARE = 0.10       # retention team can contact 10% of customers
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

## 2. Load the data
For the marker to run this notebook without any manual steps, put the CSV somewhere public
(e.g. a GitHub repo → open the file → **Raw** → copy that link) and paste the link into `DATA_URL`.
If `DATA_URL` is empty, the notebook looks for the file in the Colab session, and otherwise asks you to upload it.

In [ ]:
DATA_URL = ""                 # <- paste a public raw link to the CSV here before submitting
LOCAL_FILE = "churn Data.csv"  # name of the file if uploaded to the Colab session

def read_churn(source):
    # sep handles both comma-separated and space-separated versions of the file
    return pd.read_csv(source, sep=r"[,\t ]+", engine="python")

if DATA_URL:
    df_raw = read_churn(DATA_URL)
elif os.path.exists(LOCAL_FILE):
    df_raw = read_churn(LOCAL_FILE)
else:
    from google.colab import files
    uploaded = files.upload()
    df_raw = read_churn(list(uploaded.keys())[0])

df_raw.columns = df_raw.columns.str.strip()
print(df_raw.shape)
df_raw.head()

## 3. Clean and prepare
* **Gender** is recorded inconsistently (`F`, `female`, `Female`, `M`, `male`, `Male`). We standardise it to Male/Female,
  then encode as `Gender_Male` (1 = Male, 0 = Female).
* **IsActiveMember** is Yes/No → 1/0.
* **CustomerID** is dropped as a feature: it is just a label and carries no information about behaviour.
* No missing values, and a Balance of $0 is a genuine balance (per the brief), so no imputation.

In [ ]:
df = df_raw.copy()
print("Gender values before cleaning:", df["Gender"].value_counts().to_dict())

gender_map = {"f": "Female", "female": "Female", "m": "Male", "male": "Male"}
df["Gender"] = df["Gender"].astype(str).str.strip().str.lower().map(gender_map)
assert df["Gender"].notna().all(), "An unexpected Gender value was found - check the mapping"
print("Gender values after cleaning: ", df["Gender"].value_counts().to_dict())

df["Gender_Male"] = (df["Gender"] == "Male").astype(int)
df["IsActiveMember"] = df["IsActiveMember"].astype(str).str.strip().str.lower().map({"yes": 1, "no": 0})
assert df["IsActiveMember"].notna().all()

print("Missing values per column:", int(df.isna().sum().sum()))
print(f"Churn rate: {df['Churn'].mean():.1%}  ({df['Churn'].sum()} of {len(df)} customers)")

## 4. Features, target and train/test split
* **Target:** `Churn` (1 = churned, 0 = retained) → binary classification.
* **Split:** 80% train / 20% test, **stratified** so both sets keep the same ~18% churn rate.
  The test set is only used once, at the end, to report results.

In [ ]:
FEATURES = ["Age", "Gender_Male", "TenureMonths", "NumProducts", "IsActiveMember",
            "Balance", "NumTransactions_Last6Months", "ComplaintsLast6Months"]
TARGET = "Churn"

X = df[FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE)

base_rate = y_test.mean()
n_contact_test = int(round(CONTACT_SHARE * len(y_test)))
print(f"Train: {len(X_train)} rows, churn {y_train.mean():.1%}")
print(f"Test:  {len(X_test)} rows, churn {base_rate:.1%}")
print(f"Top-10% of the test set = {n_contact_test} customers")

## 5. Evaluation helper
Accuracy alone is misleading here: predicting "nobody churns" is already ~82% accurate but finds zero churners.
So for every model we report:
* **Recall** – share of actual churners the model flags (at the default 0.5 cut-off)
* **Precision** – of the customers flagged, how many really churn
* **F1** – balance of precision and recall
* **ROC-AUC** – how well the model *ranks* churners above non-churners (0.5 = random, 1.0 = perfect)
* **Precision @ top 10%** – if we call the 10% highest-risk customers, what share are real churners (this is the business metric)
* **Churners captured @ top 10%** – what share of all churners are inside that 10%
* **Lift** – how many times better than calling customers at random

In [ ]:
def top_k_metrics(y_true, scores, share=CONTACT_SHARE):
    y_true = np.asarray(y_true)
    k = int(round(share * len(y_true)))
    top_idx = np.argsort(-np.asarray(scores), kind="stable")[:k]
    hits = y_true[top_idx].sum()
    prec_k = hits / k
    return {"Precision@10%": prec_k,
            "Churners captured@10%": hits / y_true.sum(),
            "Lift@10%": prec_k / y_true.mean(),
            "Churners in top 10%": f"{hits} of {k}"}

def evaluate(name, y_true, proba, threshold=0.5):
    pred = (np.asarray(proba) >= threshold).astype(int)
    row = {"Model": name,
           "Accuracy": accuracy_score(y_true, pred),
           "Precision": precision_score(y_true, pred, zero_division=0),
           "Recall": recall_score(y_true, pred, zero_division=0),
           "F1": f1_score(y_true, pred, zero_division=0),
           "ROC-AUC": roc_auc_score(y_true, proba)}
    row.update(top_k_metrics(y_true, proba))
    return row

results = []      # every model adds one row here; teammates can append theirs (e.g. logistic regression)
test_scores = {}  # predicted churn probabilities on the test set, for the charts below

## 6. Baselines
1. **Majority class** – always predict "stays". Shows why accuracy is misleading.
2. **Random selection** – calling a random 10% would find churners at the base rate (~18%).
3. **Simple business rule** – call the customers with the **shortest tenure** first
   (tenure was the biggest difference in our Task 1 exploration). A model is only worth using if it beats this.

In [ ]:
dummy = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
dummy_proba = dummy.predict_proba(X_test)[:, 1]
row = evaluate("Baseline: majority class", y_test, dummy_proba)
row["ROC-AUC"] = 0.5
row.update({"Precision@10%": base_rate, "Churners captured@10%": CONTACT_SHARE, "Lift@10%": 1.0,
            "Churners in top 10%": "random (expected)"})
results.append(row)

# Rule: shorter tenure = higher risk. Score = negative tenure, so the newest customers rank first.
rule_score = -X_test["TenureMonths"].values
rule_row = {"Model": "Baseline: shortest tenure first",
            "ROC-AUC": roc_auc_score(y_test, rule_score)}
rule_row.update(top_k_metrics(y_test, rule_score))
results.append(rule_row)
test_scores["Rule: shortest tenure"] = rule_score

pd.DataFrame(results).set_index("Model")

## 7. Model 1 – Decision Tree
**Why a decision tree:** it produces plain-English if/then rules the marketing team can read
(e.g. *"tenure under 24 months and inactive → high risk"*), captures interactions between variables automatically,
and does not need features to be scaled.

**Controlling overfitting:** an unrestricted tree memorises the training data. We tune `max_depth` (how many questions deep)
and `min_samples_leaf` (smallest group allowed at the end of a branch) with **5-fold cross-validation on the training set only**,
choosing the settings with the best ROC-AUC. `class_weight="balanced"` is also tried, so the tree pays more attention to the
minority (churn) class.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

dt_grid = GridSearchCV(
    DecisionTreeClassifier(random_state=RANDOM_STATE),
    param_grid={"max_depth": [2, 3, 4, 5, 6, 8, None],
                "min_samples_leaf": [5, 10, 20, 40],
                "class_weight": [None, "balanced"]},
    scoring="roc_auc", cv=cv, n_jobs=-1)
dt_grid.fit(X_train, y_train)

dt = dt_grid.best_estimator_
print("Best settings:", dt_grid.best_params_)
print(f"Cross-validated ROC-AUC (train folds): {dt_grid.best_score_:.3f}")

dt_proba = dt.predict_proba(X_test)[:, 1]
results.append(evaluate("Decision Tree", y_test, dt_proba))
test_scores["Decision Tree"] = dt_proba

# Overfitting check: big gap between train and test AUC = overfitting
print(f"Train ROC-AUC: {roc_auc_score(y_train, dt.predict_proba(X_train)[:, 1]):.3f} | "
      f"Test ROC-AUC: {roc_auc_score(y_test, dt_proba):.3f}")

**What the tuning found** – test ROC-AUC for each tree depth (best leaf size and class weight for each depth). This shows where extra depth stops helping.

In [ ]:
cvr = pd.DataFrame(dt_grid.cv_results_)
cvr["depth"] = cvr["param_max_depth"].astype(str)
by_depth = cvr.groupby("depth", sort=False)["mean_test_score"].max()
order = ["2", "3", "4", "5", "6", "8", "None"]
by_depth = by_depth.reindex([d for d in order if d in by_depth.index])

plt.figure(figsize=(6, 3.5))
plt.plot(by_depth.index, by_depth.values, marker="o")
plt.xlabel("max_depth"); plt.ylabel("Cross-validated ROC-AUC")
plt.title("Figure X: Decision tree – depth vs. cross-validated ROC-AUC")
plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

**The tree itself** (top 3 levels shown so it is readable – this is the kind of rule set marketing can use).

In [ ]:
plt.figure(figsize=(20, 8))
plot_tree(dt, feature_names=FEATURES, class_names=["Stays", "Churns"], filled=True,
          rounded=True, max_depth=3, fontsize=9, proportion=True)
plt.title("Figure X: Decision tree (first 3 levels)")
plt.show()

**Feature importance** – how much each variable contributes to the tree's splits (sums to 1).

In [ ]:
dt_importance = pd.Series(dt.feature_importances_, index=FEATURES).sort_values()
ax = dt_importance.plot(kind="barh", figsize=(7, 4), color="#2a7ab9")
ax.set_xlabel("Importance"); ax.set_title("Figure X: Decision tree feature importance")
plt.tight_layout(); plt.show()
dt_importance.sort_values(ascending=False)

## 8. Model 2 – k-Nearest Neighbours
**Why KNN:** it predicts a customer's risk from the most similar past customers ("customers like you who left"),
makes no assumption about the shape of the relationship, and gives a useful contrast to the rule-based tree.

**Scaling is essential:** KNN measures distance, so Balance (tens of thousands) would swamp Complaints (0–3).
`StandardScaler` puts every feature on the same scale. It sits inside a `Pipeline` so the scaler is fitted on
training data only (no leakage from the test set).

**Choosing k:** small k = noisy/overfit, large k = too smooth. We tune k (and uniform vs. distance weighting) by 5-fold cross-validation on ROC-AUC.

In [ ]:
knn_pipe = Pipeline([("scale", StandardScaler()),
                     ("knn", KNeighborsClassifier())])

knn_grid = GridSearchCV(
    knn_pipe,
    param_grid={"knn__n_neighbors": list(range(3, 82, 4)),
                "knn__weights": ["uniform", "distance"]},
    scoring="roc_auc", cv=cv, n_jobs=-1)
knn_grid.fit(X_train, y_train)

knn = knn_grid.best_estimator_
print("Best settings:", knn_grid.best_params_)
print(f"Cross-validated ROC-AUC (train folds): {knn_grid.best_score_:.3f}")

knn_proba = knn.predict_proba(X_test)[:, 1]
results.append(evaluate("k-Nearest Neighbours", y_test, knn_proba))
test_scores["KNN"] = knn_proba

print(f"Train ROC-AUC: {roc_auc_score(y_train, knn.predict_proba(X_train)[:, 1]):.3f} | "
      f"Test ROC-AUC: {roc_auc_score(y_test, knn_proba):.3f}")
if knn_grid.best_params_["knn__weights"] == "distance":
    print("Note: with distance weighting, each training customer is its own nearest neighbour, so train AUC is always ~1.0. "
          "Use the cross-validated AUC above (not train AUC) to judge overfitting for KNN.")

**Choosing k** – cross-validated ROC-AUC for each value of k.

In [ ]:
kr = pd.DataFrame(knn_grid.cv_results_)
plt.figure(figsize=(6, 3.5))
for w in ["uniform", "distance"]:
    sub = kr[kr["param_knn__weights"] == w]
    plt.plot(sub["param_knn__n_neighbors"].astype(int), sub["mean_test_score"], marker=".", label=w)
plt.axvline(knn_grid.best_params_["knn__n_neighbors"], ls="--", c="grey", lw=1)
plt.xlabel("k (number of neighbours)"); plt.ylabel("Cross-validated ROC-AUC")
plt.title("Figure X: KNN – choosing k"); plt.legend(title="weights")
plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

**Feature importance for KNN.** KNN has no built-in importance, so we use *permutation importance*:
shuffle one feature at a time on the test set and measure how much ROC-AUC drops. A bigger drop = the model relies on it more.

In [ ]:
perm = permutation_importance(knn, X_test, y_test, scoring="roc_auc",
                              n_repeats=30, random_state=RANDOM_STATE)
knn_importance = pd.Series(perm.importances_mean, index=FEATURES).sort_values()
ax = knn_importance.plot(kind="barh", figsize=(7, 4), color="#e07b39")
ax.set_xlabel("Drop in ROC-AUC when shuffled"); ax.set_title("Figure X: KNN permutation importance")
plt.tight_layout(); plt.show()
knn_importance.sort_values(ascending=False)

## 9. Model comparison

In [ ]:
comparison = pd.DataFrame(results).set_index("Model")
cols = ["Accuracy", "Precision", "Recall", "F1", "ROC-AUC",
        "Precision@10%", "Churners captured@10%", "Lift@10%", "Churners in top 10%"]
comparison = comparison[cols]
comparison

**Confusion matrices** at the default 0.5 cut-off. Note that the bank will not use a 0.5 cut-off in practice –
it will simply call the 10% highest-scoring customers – so the top-10% columns above matter more than these.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.8))
for ax, (name, proba) in zip(axes, [("Decision Tree", dt_proba), ("KNN", knn_proba)]):
    cm = confusion_matrix(y_test, (proba >= 0.5).astype(int))
    ConfusionMatrixDisplay(cm, display_labels=["Stays", "Churns"]).plot(ax=ax, colorbar=False)
    ax.set_title(name)
fig.suptitle("Figure X: Confusion matrices (test set, 0.5 cut-off)")
plt.tight_layout(); plt.show()

**ROC curves and gains chart.** The gains chart answers the business question directly:
*if we call the top X% of customers, what share of all churners do we reach?* The dashed line at 10% is the team's capacity.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

for name, s in test_scores.items():
    fpr, tpr, _ = roc_curve(y_test, s)
    axes[0].plot(fpr, tpr, label=f"{name} (AUC {roc_auc_score(y_test, s):.2f})")
axes[0].plot([0, 1], [0, 1], "k--", lw=1, label="Random (AUC 0.50)")
axes[0].set_xlabel("False positive rate"); axes[0].set_ylabel("True positive rate (recall)")
axes[0].set_title("ROC curves (test set)"); axes[0].legend(fontsize=8)

y_arr = y_test.values
for name, s in test_scores.items():
    order = np.argsort(-np.asarray(s), kind="stable")
    captured = np.cumsum(y_arr[order]) / y_arr.sum()
    pct = np.arange(1, len(y_arr) + 1) / len(y_arr)
    axes[1].plot(pct, captured, label=name)
axes[1].plot([0, 1], [0, 1], "k--", lw=1, label="Random")
axes[1].axvline(CONTACT_SHARE, color="red", ls=":", label="10% contact limit")
axes[1].set_xlabel("Share of customers contacted (highest risk first)")
axes[1].set_ylabel("Share of all churners reached")
axes[1].set_title("Gains chart (test set)"); axes[1].legend(fontsize=8)

fig.suptitle("Figure X: How well each model ranks customers by churn risk")
plt.tight_layout(); plt.show()

## 10. Responsible-use check
Gender and Age are protected/sensitive characteristics. Two checks:
1. **Does the model need Gender?** Refit the tuned decision tree and KNN without `Gender_Male` and compare ROC-AUC. If performance barely changes, the safer choice is to drop it.
2. **Who ends up on the call list?** Compare the gender and age mix of the top-10% list with the test set overall.

In [ ]:
FEATURES_NO_GENDER = [f for f in FEATURES if f != "Gender_Male"]

dt_ng = DecisionTreeClassifier(random_state=RANDOM_STATE, **dt_grid.best_params_)
dt_ng.fit(X_train[FEATURES_NO_GENDER], y_train)
knn_ng = Pipeline([("scale", StandardScaler()),
                   ("knn", KNeighborsClassifier(n_neighbors=knn_grid.best_params_["knn__n_neighbors"],
                                                weights=knn_grid.best_params_["knn__weights"]))])
knn_ng.fit(X_train[FEATURES_NO_GENDER], y_train)

print(pd.DataFrame({
    "With Gender": [roc_auc_score(y_test, dt_proba), roc_auc_score(y_test, knn_proba)],
    "Without Gender": [roc_auc_score(y_test, dt_ng.predict_proba(X_test[FEATURES_NO_GENDER])[:, 1]),
                       roc_auc_score(y_test, knn_ng.predict_proba(X_test[FEATURES_NO_GENDER])[:, 1])]},
    index=["Decision Tree", "KNN"]).round(3))

In [ ]:
def who_is_contacted(proba, name):
    k = int(round(CONTACT_SHARE * len(X_test)))
    top = X_test.iloc[np.argsort(-proba, kind="stable")[:k]]
    age_band = lambda a: pd.cut(a, [17, 39, 59, 120], labels=["18-39", "40-59", "60+"])
    return pd.DataFrame({
        f"{name}: top 10%": [top["Gender_Male"].mean(), *age_band(top["Age"]).value_counts(normalize=True).sort_index()],
        "Test set overall": [X_test["Gender_Male"].mean(), *age_band(X_test["Age"]).value_counts(normalize=True).sort_index()]},
        index=["% Male", "% age 18-39", "% age 40-59", "% age 60+"])

display(who_is_contacted(dt_proba, "Decision Tree").style.format("{:.0%}"))
display(who_is_contacted(knn_proba, "KNN").style.format("{:.0%}"))

## 11. The 10% call list
To build the actual list for the retention team, the chosen model scores **every** customer and we take the 160
(10% of 1,600) with the highest predicted churn probability.

⚠️ 1,280 of these customers were in the training data, so their scores are a little optimistic. The honest
performance figures are the **test-set** numbers in Section 9. In practice the model would be refitted on all
current data and used to score next month's customers.

Set `CHOSEN_MODEL` to whichever model the group selects.

In [ ]:
CHOSEN_MODEL = dt            # change to `knn` if the group selects KNN
CHOSEN_NAME = "Decision Tree"

scored = df[["CustomerID"] + FEATURES + [TARGET]].copy()
scored["ChurnProbability"] = CHOSEN_MODEL.predict_proba(df[FEATURES])[:, 1]
n_contact = int(round(CONTACT_SHARE * len(scored)))
call_list = (scored.sort_values(["ChurnProbability", "TenureMonths"], ascending=[False, True])
                   .head(n_contact))

print(f"{CHOSEN_NAME}: call list of {n_contact} customers")
print(f"Average profile of the call list vs. all customers:")
display(pd.DataFrame({"Call list": call_list[FEATURES].mean(),
                      "All customers": scored[FEATURES].mean()}).round(2))
call_list.head(15)

## 12. Summary for the report
The cell below prints the key numbers in one place so they can be copied straight into the report.

In [ ]:
dt_row = comparison.loc["Decision Tree"]; knn_row = comparison.loc["k-Nearest Neighbours"]
rule_row = comparison.loc["Baseline: shortest tenure first"]
print(f"Test set: {len(y_test)} customers, {int(y_test.sum())} churners (base rate {base_rate:.1%})")
print(f"Contacting top 10% = {n_contact_test} test customers (≈{n_contact} of all 1,600)\n")
for name, r in [("Random calling", None), ("Rule: shortest tenure", rule_row),
                ("Decision Tree", dt_row), ("KNN", knn_row)]:
    if r is None:
        print(f"{name:24s} precision@10% {base_rate:.0%} | lift 1.0x")
    else:
        print(f"{name:24s} precision@10% {r['Precision@10%']:.0%} | captured {r['Churners captured@10%']:.0%} "
              f"of churners | lift {r['Lift@10%']:.1f}x | ROC-AUC {r['ROC-AUC']:.3f}")
print("\nTop 3 features – Decision Tree:", ", ".join(dt_importance.sort_values(ascending=False).index[:3]))
print("Top 3 features – KNN:          ", ", ".join(knn_importance.sort_values(ascending=False).index[:3]))